# Importar Librerías y Configurar el Navegador

In [4]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from time import sleep
import csv
import os
from datetime import datetime
from random import randint
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import StaleElementReferenceException
import re

from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup, NavigableString




# ORQUESTADOR ----------------------------------------------------------------*


In [5]:
from scraping.extractor_twitter import iniciar_sesion, navegar_a_perfil, extraer_y_guardar_comentarios
from utils.helpers import generar_nombre_archivo
#from utils.helpers import configurar_log

# Configuración de logs
#configurar_log()

# FUNCIÓN PRINCIPAL
try:
    # Configuración del navegador
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)

    # Iniciar sesión
    iniciar_sesion(driver,user='juan_c.ortiz_b@uao.edu.co', pwd='3127916565', username='@OrtizBaron77043')
    #iniciar_sesion(driver,user='jamoncayop@gmail.com', pwd='Adaptiv3@*', username='@Jaime1807816689')

    
    # Navegar al perfil de @Tu_IMSS
    navegar_a_perfil(driver, "https://x.com/Tu_IMSS?f=live")
    print("¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!")

    # Extraer y guardar comentarios de los primeros tweets
    fecha_actual = datetime.now().strftime("%Y-%m-%d")
    #nombre_archivo_base = f"db/cmts_extraidos_{fecha_actual}_jaime.csv"
    nombre_archivo_base = f"db/cmts_extraidos_{fecha_actual}_camilo.csv"

    # Generar un nombre único para el archivo
    nombre_archivo = generar_nombre_archivo(nombre_archivo_base)

    # Extraer y guardar comentarios con el nuevo nombre
    extraer_y_guardar_comentarios(driver, nombre_archivo)

except Exception as e:
    print("❌ Error en la ejecución:", e)

finally:
    # Cerrar el navegador
    if driver:
        driver.quit()


✔ Correo ingresado
ℹ No se pidió confirmación adicional de nombre de usuario.
✔ Contraseña ingresada
✔ Se hizo clic en 'Iniciar sesión'
¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!
🔄 Scroll fijo #1/1
🔍 Se encontraron 6 elementos tipo tweet antes de extraer los enlaces. Jaime!
🔗 Capturadas 5 URLs de tweets (queríamos 20)

🔹 Procesando URL #1/5: https://x.com/Tu_IMSS/status/1957186969697726576

▶️  Abriendo tweet en nueva pestaña: https://x.com/Tu_IMSS/status/1957186969697726576
    🔄 Replies cargados: 0
    🔄 Replies cargados: 0
  🔍 Tamaño de page_source tras spam: 487832
  💡 Extraído: id=id_En_#Ecatepec_se_inauguró_la_UM replies=2
  👤 Autor original: IMSS  @Tu_IMSS
↪️ [guardar_comentarios] para tweet_id=id_En_#Ecatepec_se_inauguró_la_UM
🔎 oficiales: 0
🔎 spam_cells encontradas: 10, total spam artículos: 5
🔎 total candidatos tras unir: 4
→ auténticas tras filtrar y recortar a 2/2
🔄 Guardando respuesta auténtica #1
🔄 Guardando respuesta auténtica #2
  🔙 Cerrando pestaña y volvi

# Nube de palabras -------------------------------------------------------*

In [8]:
import pandas as pd
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# Leer el archivo CSV
#df = pd.read_csv("db/comentarios_imss_urls_1333333.csv")
df = pd.read_csv("db/db_procesadas/comentarios_limpios_reemplazado2.csv", on_bad_lines='skip')


# Combinar todos los comentarios
texto = " ".join(str(c) for c in df["comentario"].dropna())

# Stopwords en español
stopwords = set(STOPWORDS)
stopwords.update(["https", "rt", "t.co", "si", "no", "tener", "ser", "estar", "del", "por", "para", "con", "más", "menos","hay","de","la","en","y","las",
                  "su","se","lo","el","sin","que","un","una","mi","al","comentario","sus","los","tiene","ya","e","es"])

# Crear y mostrar la nube de palabras
wordcloud = WordCloud(width=1000, height=500, background_color="white", stopwords=stopwords, collocations=False).generate(texto)

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Nube de Palabras - Comentarios", fontsize=20)
plt.show()


KeyboardInterrupt: 

# Parte 2 Análisis de sentimientos ---------------*


### Ejemplo sencillo ***

In [ ]:
import pandas as pd
import subprocess
import json
from limpieza.preprocesamiento import limpiar_comentario, normalizar_comillas, limpiar_respuesta_ollama

def limpiar_respuesta_ollama(respuesta):
    # Reemplazar las claves incorrectas (en este caso "sentimento" por "sentimiento")
    respuesta_corregida = respuesta.replace('"sentimento"', '"sentimiento"')
    
    try:
        # Intentar parsear el JSON corregido
        return json.loads(respuesta_corregida)
    except json.JSONDecodeError as e:
        print(f"[ERROR] Error al parsear el JSON: {e}")
        return {"sentimiento": "error", "categoria": "error"}

def probar_ollama_dataset(indice=10):
    """
    Carga un comentario específico del dataset y lo envía a Ollama para clasificarlo.
    """
    print("📥 [CARGANDO] Leyendo CSV...")
    df = pd.read_csv("db/comentarios_imss_urls_12.csv")
    #df = pd.read_csv("db/db_procesadas/comentarios_limpios_reemplazado2.csv", on_bad_lines='skip')
    print(f"🔢 [INFO] Total de comentarios: {len(df)}")

    comentario_original = str(df["comentario"].iloc[indice])

    comentario_limpio = limpiar_comentario(normalizar_comillas(comentario_original))

    prompt = f"""
Eres un experto en análisis de sentimientos en español de México. 
Debes interpretar lenguaje coloquial, slang, modismos y expresiones culturales comunes en México, 
incluyendo sarcasmo, ironía, abreviaciones, errores ortográficos y frases indirectas.

Reglas de interpretación:
- Si hay sarcasmo negativo, clasifícalo como "negativo".
- Si hay mezcla de elogio y queja, elige la que tenga más peso emocional.
- Si el mensaje es solo informativo o no transmite emoción clara, es "neutro" y "otro".
- Usa el contexto implícito y el sentido común para interpretar.

Definiciones:
- Sentimiento:
  - "positivo": expresa aprobación, alegría o gratitud genuina.
  - "negativo": expresa frustración, enojo, burla o crítica.
  - "neutro": informativo o sin carga emocional clara.
- Categoría:
  - "queja": expresa reclamo o insatisfacción.
  - "elogio": expresa reconocimiento o agradecimiento.
  - "otro": no encaja en las anteriores (preguntas, info neutral, comentarios irrelevantes).

Ejemplos:
Comentario: "Qué chido el servicio del IMSS, me atendieron rapidísimo."
Respuesta: {{"sentimiento": "positivo", "categoria": "elogio"}}

Comentario: "No mames, otra vez sin citas disponibles, qué chafa."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "El IMSS abre a las 8 AM."
Respuesta: {{"sentimiento": "neutro", "categoria": "otro"}}

Comentario: "Gracias por el apoyo, pero podrían mejorar las esperas."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "Uy sí, rapidísimo como siempre... tres horas esperando."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Analiza el siguiente comentario:
"{comentario_limpio}"

Devuelve únicamente un JSON válido con esta estructura exacta:
{{
  "sentimiento": "positivo" | "negativo" | "neutro",
  "categoria": "queja" | "elogio" | "otro"
}}
No incluyas explicaciones, comentarios ni bloques de código.
    """

    print(" [TEST] Enviando prompt a Ollama...")
    try:
        proceso = subprocess.Popen(
            ["ollama", "run", "mistral"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace"
        )
        salida, error = proceso.communicate(prompt, timeout=180)
        #salida, error = proceso.communicate(prompt, timeout=60)
        print("[DEBUG] STDOUT:", salida.strip())
        print("[DEBUG] STDERR:", error.strip() if error else "VACÍO")

        # Limpiar la respuesta y devolver el resultado
        return limpiar_respuesta_ollama(salida)
    except Exception as e:
        print(f" [ERROR] {e}")
        return {"sentimiento": "error", "categoria": "error"}

if __name__ == "__main__":
    resultado = probar_ollama_dataset(indice=2)  # Cambia el índice para otro comentario
    print(" [RESULTADO FINAL]:", resultado)


📥 [CARGANDO] Leyendo CSV...
🔢 [INFO] Total de comentarios: 24
 [TEST] Enviando prompt a Ollama...
 [ERROR] Command '['ollama', 'run', 'mistral']' timed out after 180 seconds
 [RESULTADO FINAL]: {'sentimiento': 'error', 'categoria': 'error'}


### Ejemplo con listado iterando cada comentario ***

In [ ]:
import pandas as pd
import subprocess
import json
from limpieza.preprocesamiento import limpiar_comentario, normalizar_comillas, limpiar_respuesta_ollama

def clasificar_dataset_completo():
    """
    Itera sobre todos los comentarios del dataset, los envía a Ollama para clasificar
    y guarda el resultado en un nuevo CSV con la respuesta cruda.
    """
    print("📥 [CARGANDO] Leyendo CSV...")
    df = pd.read_csv("db/comentarios_imss_urls_12.csv")
    print(f"🔢 [INFO] Total de comentarios: {len(df)}")

    resultados = []
    respuestas_crudas = []

    for i, comentario in enumerate(df["comentario"]):
        comentario_original = str(comentario)
        comentario_limpio = limpiar_comentario(normalizar_comillas(comentario_original))

        print(f"📝 [{i+1}/{len(df)}] Comentario limpio: {comentario_limpio[:50]}...")

        prompt = f"""
Eres un experto en análisis de sentimientos en español de México. 
Debes interpretar lenguaje coloquial, slang, modismos y expresiones culturales comunes en México, 
incluyendo sarcasmo, ironía, abreviaciones, errores ortográficos y frases indirectas.

Reglas de interpretación:
- Si hay sarcasmo negativo, clasifícalo como "negativo".
- Si hay mezcla de elogio y queja, elige la que tenga más peso emocional.
- Si el mensaje es solo informativo o no transmite emoción clara, es "neutro" y "otro".
- Usa el contexto implícito y el sentido común para interpretar.

Definiciones:
- Sentimiento:
  - "positivo": expresa aprobación, alegría o gratitud genuina.
  - "negativo": expresa frustración, enojo, burla o crítica.
  - "neutro": informativo o sin carga emocional clara.
- Categoría:
  - "queja": expresa reclamo o insatisfacción.
  - "elogio": expresa reconocimiento o agradecimiento.
  - "otro": no encaja en las anteriores (preguntas, info neutral, comentarios irrelevantes).

Ejemplos:
Comentario: "Qué chido el servicio del IMSS, me atendieron rapidísimo."
Respuesta: {{"sentimiento": "positivo", "categoria": "elogio"}}

Comentario: "No mames, otra vez sin citas disponibles, qué chafa."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "El IMSS abre a las 8 AM."
Respuesta: {{"sentimiento": "neutro", "categoria": "otro"}}

Comentario: "Gracias por el apoyo, pero podrían mejorar las esperas."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "Uy sí, rapidísimo como siempre... tres horas esperando."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Analiza el siguiente comentario:
"{comentario_limpio}"

Devuelve únicamente un JSON válido con esta estructura exacta:
{{
  "sentimiento": "positivo" | "negativo" | "neutro",
  "categoria": "queja" | "elogio" | "otro"
}}
No incluyas explicaciones, comentarios ni bloques de código.
        """

        try:
            proceso = subprocess.Popen(
                ["ollama", "run", "mistral"],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
                errors="replace"
            )
            salida, error = proceso.communicate(prompt, timeout=180)
            salida = salida.strip() if salida else ""
            respuestas_crudas.append(salida)

            print("[DEBUG] STDOUT:", salida)
            if error:
                print("[DEBUG] STDERR:", error.strip())

            analisis = limpiar_respuesta_ollama(salida)
        except Exception as e:
            print(f"❌ [ERROR en comentario {i+1}]: {e}")
            analisis = {"sentimiento": "error", "categoria": "error"}
            respuestas_crudas.append("")

        resultados.append(analisis)

    # Guardar resultados
    df["respuesta_raw"] = respuestas_crudas
    df["sentimiento"] = [r.get("sentimiento", "error") for r in resultados]
    df["categoria"] = [r.get("categoria", "error") for r in resultados]

    output_file = "db/db_procesadas/tweets_clasificados.csv"
    df.to_csv(output_file, index=False, encoding="utf-8")
    print(f"🎉 [COMPLETADO] Resultados guardados en: {output_file}")

if __name__ == "__main__":
    clasificar_dataset_completo()




📥 [CARGANDO] Leyendo CSV...
🔢 [INFO] Total de comentarios: 24
📝 [1/24] Comentario limpio: mujer en condición de calle ataca a trabajadores d...
❌ [ERROR en comentario 1]: Command '['ollama', 'run', 'mistral']' timed out after 180 seconds
📝 [2/24] Comentario limpio: nuestro es el encargado de las múltiples funciones...
[DEBUG] STDOUT: {"sentimiento": "positivo", "categoria": "otro"}
[DEBUG] STDERR: ⠙ ⠹ ⠸ ⠼ ⠼ ⠦ ⠦ ⠇ ⠏ ⠋ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠏ ⠏ ⠋ ⠙ ⠹ ⠸ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠋ ⠹ ⠸ ⠸ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠹ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠴ ⠴ ⠦ ⠇ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠸ ⠸ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠸ ⠼ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠋ ⠹ ⠹ ⠸ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠴ ⠦ ⠧ ⠧ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠧ ⠇ ⠋ ⠙ ⠹ ⠸ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠋ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠧ ⠧ ⠏ ⠋ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠇ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠴ ⠦ ⠦ ⠧ ⠏ ⠋ ⠙ ⠹ ⠹ ⠼ ⠼ ⠴ ⠧ ⠇ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠦ ⠇ ⠏ ⠋ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠏ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠴ ⠦ ⠇ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠋ ⠙ ⠙ ⠸ ⠼ ⠴ ⠴ ⠦ ⠇ ⠏ ⠋ ⠙ ⠹ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ ⠙ ⠹ ⠹ ⠼ ⠴ ⠦ ⠧ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠼ ⠴ ⠦ ⠦ ⠧ ⠇ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠦ ⠧ ⠧ ⠇ ⠋ ⠋ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠹ ⠸ ⠴ ⠦ ⠦ ⠧ ⠇ ⠏

# PRueba funcional extrayendo CSV

In [1]:
import pandas as pd
import subprocess
import json
import datetime
from limpieza.preprocesamiento import limpiar_comentario, normalizar_comillas, limpiar_respuesta_ollama

def clasificar_dataset_completo():
    """
    Itera sobre todos los comentarios del dataset, los envía a Ollama para clasificar
    y guarda el resultado en un nuevo CSV y también en Excel (.xlsx).
    """
    print("📥 [CARGANDO] Leyendo CSV...")
    df = pd.read_csv("db/comentarios_imss_urls_12.csv")
    print(f"🔢 [INFO] Total de comentarios: {len(df)}")

    resultados = []
    respuestas_crudas = []

    for i, comentario in enumerate(df["comentario"]):
        comentario_original = str(comentario)
        comentario_limpio = limpiar_comentario(normalizar_comillas(comentario_original))

        print(f"📝 [{i+1}/{len(df)}] Comentario limpio: {comentario_limpio[:50]}...")

        prompt = f"""
Eres un experto en análisis de sentimientos en español de México. 
Debes interpretar lenguaje coloquial, slang, modismos y expresiones culturales comunes en México, 
incluyendo sarcasmo, ironía, abreviaciones, errores ortográficos y frases indirectas.

Reglas de interpretación:
- Si hay sarcasmo negativo, clasifícalo como "negativo".
- Si hay mezcla de elogio y queja, elige la que tenga más peso emocional.
- Si el mensaje es solo informativo o no transmite emoción clara, es "neutro" y "otro".
- Usa el contexto implícito y el sentido común para interpretar.

Definiciones:
- Sentimiento:
  - "positivo": expresa aprobación, alegría o gratitud genuina.
  - "negativo": expresa frustración, enojo, burla o crítica.
  - "neutro": informativo o sin carga emocional clara.
- Categoría:
  - "queja": expresa reclamo o insatisfacción.
  - "elogio": expresa reconocimiento o agradecimiento.
  - "otro": no encaja en las anteriores (preguntas, info neutral, comentarios irrelevantes).

Ejemplos:
Comentario: "Qué chido el servicio del IMSS, me atendieron rapidísimo."
Respuesta: {{"sentimiento": "positivo", "categoria": "elogio"}}

Comentario: "No mames, otra vez sin citas disponibles, qué chafa."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "El IMSS abre a las 8 AM."
Respuesta: {{"sentimiento": "neutro", "categoria": "otro"}}

Comentario: "Gracias por el apoyo, pero podrían mejorar las esperas."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "Uy sí, rapidísimo como siempre... tres horas esperando."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Analiza el siguiente comentario:
"{comentario_limpio}"

Devuelve únicamente un JSON válido con esta estructura exacta:
{{
  "sentimiento": "positivo" | "negativo" | "neutro",
  "categoria": "queja" | "elogio" | "otro"
}}
No incluyas explicaciones, comentarios ni bloques de código.
        """

        try:
            proceso = subprocess.Popen(
                ["ollama", "run", "mistral"],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
                errors="replace"
            )
            salida, error = proceso.communicate(prompt, timeout=180)
            salida = salida.strip() if salida else ""
            respuestas_crudas.append(salida)

            print("[DEBUG] STDOUT:", salida)
            if error:
                print("[DEBUG] STDERR:", error.strip())

            analisis = limpiar_respuesta_ollama(salida)
        except Exception as e:
            print(f"❌ [ERROR en comentario {i+1}]: {e}")
            analisis = {"sentimiento": "error", "categoria": "error"}
            respuestas_crudas.append("")

        resultados.append(analisis)

    # Guardar resultados
    df["respuesta_raw"] = respuestas_crudas
    df["sentimiento"] = [r.get("sentimiento", "error") for r in resultados]
    df["categoria"] = [r.get("categoria", "error") for r in resultados]

    # Crear timestamp para no sobrescribir
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_csv = f"db/db_procesadas/tweets_clasificados_{timestamp}.csv"
    output_xlsx = f"db/db_procesadas/tweets_clasificados_{timestamp}.xlsx"

    # Guardar en CSV y Excel
    df.to_csv(output_csv, index=False, encoding="utf-8")
    df.to_excel(output_xlsx, index=False, engine="openpyxl")

    print(f"🎉 [COMPLETADO] Resultados guardados en:\n  📄 CSV : {output_csv}\n  📊 Excel : {output_xlsx}")

if __name__ == "__main__":
    clasificar_dataset_completo()


📥 [CARGANDO] Leyendo CSV...
🔢 [INFO] Total de comentarios: 24
📝 [1/24] Comentario limpio: mujer en condición de calle ataca a trabajadores d...
❌ [ERROR en comentario 1]: Command '['ollama', 'run', 'mistral']' timed out after 180 seconds
📝 [2/24] Comentario limpio: nuestro es el encargado de las múltiples funciones...
❌ [ERROR en comentario 2]: Command '['ollama', 'run', 'mistral']' timed out after 180 seconds
📝 [3/24] Comentario limpio: y siguen de voceros, pero de resultados nada. para...
[DEBUG] STDOUT: {"sentimiento": "negativo", "categoria": "queja"}
[DEBUG] STDERR: ⠙ ⠙ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠹ ⠼ ⠴ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠦ ⠦ ⠧ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠼ ⠴ ⠧ ⠇ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠦ ⠧ ⠧ ⠇ ⠏ ⠙ ⠹ ⠹ ⠸ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠋ ⠹ ⠸ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠋ ⠙ ⠹ ⠹ ⠸ ⠴ ⠦ ⠧ ⠧ ⠏ ⠋ ⠋ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠋ ⠙ ⠹ ⠼ ⠴ ⠦ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠋ ⠙ ⠹ ⠸ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠹ ⠸ ⠴ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠹ ⠸ ⠸ ⠴ ⠴ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠹ ⠸ ⠴ ⠦ ⠧ ⠧ ⠏ ⠋ ⠙ ⠹ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠏ ⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠋ ⠙ ⠹ 

ModuleNotFoundError: No module named 'openpyxl'

# Pruebas


OBJETIVO de la prueba
Validar si n8n puede ejecutar un script Python con Selenium en tu máquina (Windows) de forma correcta desde el nodo Execute Command.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.google.com")
time.sleep(5)
driver.quit()


# Prompt enviado a Mistral

In [ ]:
prompt = f"""
Eres un experto en análisis de sentimientos en español de México. 
Debes interpretar lenguaje coloquial, slang, modismos y expresiones culturales comunes en México, 
incluyendo sarcasmo, ironía, abreviaciones, errores ortográficos y frases indirectas.

Reglas de interpretación:
- Si hay sarcasmo negativo, clasifícalo como "negativo".
- Si hay mezcla de elogio y queja, elige la que tenga más peso emocional.
- Si el mensaje es solo informativo o no transmite emoción clara, es "neutro" y "otro".
- Usa el contexto implícito y el sentido común para interpretar.

Definiciones:
- Sentimiento:
  - "positivo": expresa aprobación, alegría o gratitud genuina.
  - "negativo": expresa frustración, enojo, burla o crítica.
  - "neutro": informativo o sin carga emocional clara.
- Categoría:
  - "queja": expresa reclamo o insatisfacción.
  - "elogio": expresa reconocimiento o agradecimiento.
  - "otro": no encaja en las anteriores (preguntas, info neutral, comentarios irrelevantes).

Ejemplos:
Comentario: "Qué chido el servicio del IMSS, me atendieron rapidísimo."
Respuesta: {{"sentimiento": "positivo", "categoria": "elogio"}}

Comentario: "No mames, otra vez sin citas disponibles, qué chafa."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "El IMSS abre a las 8 AM."
Respuesta: {{"sentimiento": "neutro", "categoria": "otro"}}

Comentario: "Gracias por el apoyo, pero podrían mejorar las esperas."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Comentario: "Uy sí, rapidísimo como siempre... tres horas esperando."
Respuesta: {{"sentimiento": "negativo", "categoria": "queja"}}

Analiza el siguiente comentario:
"{comentario_limpio}"

Devuelve únicamente un JSON válido con esta estructura exacta:
{{
  "sentimiento": "positivo" | "negativo" | "neutro",
  "categoria": "queja" | "elogio" | "otro"
}}
No incluyas explicaciones, comentarios ni bloques de código.
        """
